# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**The Contract (5 Answers):**
1. **One row means:** One unique content item for a specific client (`client_hash_id` + `content_hash_id`).
2. **Tables used:** `fact_content_daily_performance` for performance metrics, and `dim_content` for content metadata.
3. **Time window:** A mid-panel training month (e.g., March 2026), specifically using the past 30 days as features and the next 15 days as the target window.
4. **What we predict:** A binary label (`dropped_traffic_next15d`) indicating if the page loses more than 15% of its impressions in the target window.
5. **Deliberately excluded:** `ga4_sessions` (because it is missing for a large portion of our clients, so we cannot safely rely on it for global predictions without introducing missingness bias).


In [4]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

load_dotenv('../../.env')
HF_TOKEN = os.environ.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MID_PANEL = f"(SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31')"


## 3. Verify it with queries (grain, counts, missing values, windows)


In [5]:
# 1. Grain Check
print("--- Grain Check ---")
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {MID_PANEL}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicates found violating grain: {len(grain_check)}")

# 2. Row count and date span
print("\n--- Row Count & Span ---")
span = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM {MID_PANEL}
""").df()
print(f"Total Rows: {span['total_rows'][0]}")
print(f"Window: {span['start_date'][0]} to {span['end_date'][0]}")

# 3. Availability check with IS TRUE
print("\n--- Availability Filter ---")
ga4_avail = con.sql(f"""
    SELECT COUNT(*) as valid_rows
    FROM {MID_PANEL}
    WHERE ga4_data_available IS TRUE
""").df()
print(f"Rows surviving GA4 availability filter: {ga4_avail['valid_rows'][0]}")


--- Grain Check ---
Duplicates found violating grain: 0

--- Row Count & Span ---
Total Rows: 9841378
Window: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

--- Availability Filter ---
Rows surviving GA4 availability filter: 413966


In [6]:
# Build 5-feature frame and the Deliberate Leakage Trap
print("--- Building Feature Frame ---")
query = f"""
    WITH bounds AS (SELECT CAST('2026-03-15' AS DATE) AS split_d),
    agg AS (
        SELECT 
            f.client_hash_id, 
            f.content_hash_id,
            SUM(CASE WHEN f.report_date <= b.split_d THEN f.gsc_impressions ELSE 0 END) AS imp_past,
            AVG(CASE WHEN f.report_date <= b.split_d THEN f.gsc_avg_position END) AS pos_past,
            SUM(CASE WHEN f.report_date <= b.split_d THEN f.gsc_clicks ELSE 0 END) AS clicks_past,
            SUM(CASE WHEN f.report_date <= b.split_d THEN f.ga4_sessions ELSE 0 END) AS sessions_past,
            SUM(CASE WHEN f.report_date > b.split_d THEN f.gsc_impressions ELSE 0 END) AS imp_future
        FROM {MID_PANEL} f
        CROSS JOIN bounds b
        GROUP BY 1, 2
        HAVING imp_past >= 100
    )
    SELECT 
        imp_past, pos_past, clicks_past, sessions_past,
        COALESCE(clicks_past, 0) / NULLIF(imp_past, 0) AS ctr_past,
        imp_future,
        CASE WHEN imp_future < imp_past * 0.85 THEN 1 ELSE 0 END AS dropped_traffic
    FROM agg
"""
df = con.sql(query).df().fillna(0)

# Feature Explanations
print("Features built:")
print("1. imp_past: Knowable at the decision moment because it aggregates only dates before the split.")
print("2. pos_past: Knowable at the decision moment because it averages positions only before the split.")
print("3. clicks_past: Knowable at the decision moment because it sums clicks only before the split.")
print("4. sessions_past: Knowable at the decision moment because it tracks sessions only before the split.")
print("5. ctr_past: Knowable at the decision moment because it is derived mathematically from past clicks and past impressions.")

# The Trap!
print("\n--- The Leakage Trap ---")
X_leaky = df[['imp_past', 'pos_past', 'clicks_past', 'sessions_past', 'ctr_past', 'imp_future']] # TRAP!
X_honest = df[['imp_past', 'pos_past', 'clicks_past', 'sessions_past', 'ctr_past']]
y = df['dropped_traffic']

X_l_train, X_l_test, X_h_train, X_h_test, y_train, y_test = train_test_split(X_leaky, X_honest, y, test_size=0.3, random_state=42)

clf_leaky = DecisionTreeClassifier(max_depth=5).fit(X_l_train, y_train)
clf_honest = DecisionTreeClassifier(max_depth=5).fit(X_h_train, y_train)

print(f"Accuracy WITH the future label-derived column (Leaky): {accuracy_score(y_test, clf_leaky.predict(X_l_test)):.1%}")
print(f"Accuracy WITHOUT the future column (Honest): {accuracy_score(y_test, clf_honest.predict(X_h_test)):.1%}")



--- Building Feature Frame ---
Features built:
1. imp_past: Knowable at the decision moment because it aggregates only dates before the split.
2. pos_past: Knowable at the decision moment because it averages positions only before the split.
3. clicks_past: Knowable at the decision moment because it sums clicks only before the split.
4. sessions_past: Knowable at the decision moment because it tracks sessions only before the split.
5. ctr_past: Knowable at the decision moment because it is derived mathematically from past clicks and past impressions.

--- The Leakage Trap ---
Accuracy WITH the future label-derived column (Leaky): 83.4%
Accuracy WITHOUT the future column (Honest): 67.8%


## 4. Data limits

*   **The Final Month Sample Warning:** We cannot use the `_sample` table (June 2026) to develop our label logic because it represents the absolute end of the timeline. If we use the last month to engineer features predicting the future, there is no future data left to test on! We must treat June 2026 as a sealed holdout and use mid-panel months (like March 2026) for all feature engineering and training.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
